# 數位控制系統第二章：離散時間系統與 z 轉換（教學版 Notebook）

本 Notebook 是 `chp2.md` 教材的教學版，額外補充：
- 每個第一次出現的 MATLAB / Octave 函數（`ztrans`, `iztrans`, `residue`, `conv`, `eig`, `ss2tf`, ...）的逐步解說
- 每個公式的推導過程與符號定義
- 公式 → 程式碼的逐項對照
- 可直接執行的程式碼範例

建議搭配 `chp2.md`（完整理論與推導）與 `chp2.m`（精簡可執行版）一起閱讀。

---

## 🔧 環境設定

在 Octave Jupyter Notebook 中繪圖與使用符號運算工具箱，需要：
1. 使用 `graphics_toolkit('gnuplot')` 讓圖表能內嵌顯示
2. 載入 `control` 套件（提供 `tf`, `ss`, `ss2tf` 等）與 `symbolic` 套件（提供 `syms`, `ztrans`, `residue` 等）
3. 設定中文字型以避免亂碼

> 若你是在標準 MATLAB（非 Octave）環境執行，`pkg load ...` 這幾行會報錯，直接刪除或註解掉即可 —— MATLAB 內建 Control System Toolbox 與 Symbolic Math Toolbox，不需要額外載入套件。

In [ ]:
%plot --format svg
% 上面這行是 Octave Jupyter kernel 的「magic」語法（必須放在 cell 第一行），
% 讓圖表以 SVG 格式內嵌顯示，SVG 對中文字型的支援較好，可避免圖表中的中文標題/座標軸文字亂碼。

warning('off', 'Octave:graphics-toolkit');
graphics_toolkit('gnuplot');
clear; clc;
pkg load control;    % 若在 MATLAB 中執行，請刪除或註解此行
pkg load symbolic;   % 若在 MATLAB 中執行，請刪除或註解此行

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

## 一、離散時間系統的基本概念 (2.1–2.2)

**連續時間系統**由微分方程式描述；**離散時間系統**由**差分方程式**（difference equation）描述——因為數位電腦只能在固定取樣瞬間 $kT$（$k=0,1,2,\dots$）讀取與輸出數值。

一個典型的例子：把類比 PI 控制器 $m(t)=K_Pe(t)+K_I\int_0^t e(\tau)d\tau$ 數位化。用矩形法則數值積分近似積分項：

$$
x(kT) = x[(k-1)T] + Te(kT)
$$

這是一個**一階差分方程式**。推廣到 $n$ 階線性非時變差分方程式的一般形式：

$$
x(k) = b_n e(k) + b_{n-1}e(k-1) + \cdots + b_0 e(k-n) - a_{n-1}x(k-1) - \cdots - a_0 x(k-n)
$$

**符號定義**：

| 符號 | 意義 |
|---|---|
| $T$ | 取樣週期（秒），即取樣瞬間之間的時間間隔 |
| $k$ | 取樣時刻的整數編號，實際時間為 $kT$ |
| $a_i, b_i$ | 差分方程式（數位濾波器）的係數 |
| $n$ | 差分方程式的階數 |

> **觀念解析**：把 $T$ 決定得夠小、把差分方程式的係數設計得當，數位濾波器的行為就能逼近原本的類比濾波器——這正是「用數位電腦取代類比電路」的數學基礎。

## 二、z 轉換的定義 (2.3)

正如拉普拉斯轉換把微分方程式變成代數方程式，**z 轉換**把差分方程式變成代數方程式。z 轉換是對**數列** $\{e(k)\}$ 定義的：

$$
E(z) = \mathcal{Z}[\{e(k)\}] = \sum_{k=0}^{\infty} e(k)z^{-k}
$$

也就是說，$E(z)$ 是變數 $z^{-1}$ 的冪級數，**級數的每一項係數就是數列 $e(k)$ 的值**。這個「係數＝數列值」的性質，是整章 z 轉換理論的出發點。

### 公式 → 程式碼：`ztrans()`

MATLAB/Octave 的 `ztrans(expr)` 對符號表達式（以 `k` 為自變數的數列）直接計算 z 轉換，等同於自動完成上面的無窮級數求和。使用前要先用 `syms` 宣告符號變數。

In [ ]:
syms k a T z

% 例 2.2：e(k)=1（所有 k），求 E(z)
Ez_step = ztrans(1^k)
% 理論結果：E(z) = z/(z-1)

**函數介紹：`syms`**——宣告接下來出現的變數為「符號變數」，讓 MATLAB 做符號（代數）運算而非數值運算，是 Symbolic Math Toolbox 的起手式。

**函數介紹：`ztrans(expr)`**——對符號表達式 `expr`（以 `k` 為自變數）計算單邊 z 轉換 $E(z)=\sum_{k=0}^\infty e(k)z^{-k}$，回傳值以 `z` 表示。

In [ ]:
% 例 2.3：e(k) = a^(kT)，求 E(z)
Ez_exp = ztrans(exp(-a*k*T))
% 理論結果：E(z) = z/(z - exp(-a*T))，因為 a^T = exp(T*ln(a)) = exp(-a*T)（此處取 e(k)=exp(-a*kT) 為例）

**數學 ↔ MATLAB 對照**：

- 數學上的封閉形式 $E(z)=\dfrac{z}{z-a^T}$（由等比級數公式 $\frac{1}{1-x}=1+x+x^2+\cdots$，取 $x=a^Tz^{-1}$ 得到）
- ↔ MATLAB 輸出 `z/(z - exp(-a*T))`（符號引擎習慣用指數形式 $e^{-aT}$ 表示 $a^T$）

---

## 三、z 轉換的性質 (2.4)

### 1. 線性、實數平移、複數平移

| 性質 | 公式 |
|---|---|
| 線性 | $\mathcal{Z}[\alpha e_1(k)+\beta e_2(k)] = \alpha E_1(z)+\beta E_2(z)$ |
| 右移（延遲） | $\mathcal{Z}[e(k-n)u(k-n)] = z^{-n}E(z)$ |
| 左移（超前） | $\mathcal{Z}[e(k+n)u(k)] = z^n\left[E(z)-\sum_{k=0}^{n-1}e(k)z^{-k}\right]$ |
| 複數平移 | $\mathcal{Z}[a^{kT}e(kT)] = E(z)\big\vert_{z\to z/a^T}$ |

**物理意義**：右移不遺失資訊所以形式簡單；左移會把最前面 $n$ 個值「甩出視窗」，公式中要扣掉這些即將消失的項來補償。複數平移則是「時域乘上指數」對應「z 域座標縮放」。

### 2. 初值定理與終值定理

$$
e(0) = \lim_{z\to\infty}E(z), \qquad \lim_{n\to\infty}e(n) = \lim_{z\to1}(z-1)E(z)
$$

終值定理成立的條件：$E(z)$ 所有極點在單位圓內，最多允許在 $z=1$ 處有一個單重極點（第 7 章會證明）。

In [ ]:
% 例 2.6：複數平移定理 -> kT * a^(kT)
ekT = (k*T)*exp(a*k*T);
Ez_complex_shift = ztrans(ekT)
pretty(Ez_complex_shift)
% 理論結果：(T*z*exp(T*a)) / (z - exp(T*a))^2

**函數介紹：`pretty(expr)`**——把符號表達式排版成較接近手寫數學的形式（分數用橫線顯示），純粹是顯示用途，方便閱讀複雜結果，不影響運算。

---

## 四、由 s 域函數求 z 轉換 (2.5) —— 例 2.9

當系統函數以 $s$ 域（連續時間）給出時，標準做法是：**先做部分分式展開，再逐項查表轉成 z 域**。

求 $E(s)=\dfrac{s^2+4s+3}{s^3+6s^2+8s}=\dfrac{s^2+4s+3}{s(s+2)(s+4)}$ 的 z 轉換。

部分分式展開（用留數法 $K_i=(s-p_i)E(s)|_{s=p_i}$）：

$$
E(s) = \frac{0.375}{s}+\frac{0.25}{s+2}+\frac{0.375}{s+4}
$$

> ⚠️ 原書 OCR 文字把中間係數印成 0.025，經重新驗算應為 **0.25**，此處已訂正。

逐項用「$\frac{1}{s+a}$ 取樣後對應 $\frac{z}{z-e^{-aT}}$」查表轉成 z 域，再相加通分即得 $E(z)$。

In [ ]:
% 方法一：手動走一遍 residue -> z 域極點 -> 合併多項式
T = 0.1;
num = [1 4 3];
denom = [1 6 8 0];              % s(s+2)(s+4)，注意不可有重根
n = length(denom);
Es = tf(num, denom)              % 先建立 s 域轉移函數方便檢查

**函數介紹：`tf(num, den)`**——建立連續時間轉移函數模型，`num`、`den` 為多項式係數向量（由高次到低次）。

In [ ]:
[r, p, kdir] = residue(num, denom);   % 部分分式展開：留數 r、極點 p、直接項 kdir
r, p, kdir

**函數介紹：`[r,p,k] = residue(num, den)`**——對有理多項式 $\dfrac{\text{num}(s)}{\text{den}(s)}$ 做部分分式展開，回傳：
- `r`：留數（residues），對應每個極點的分子係數
- `p`：極點（poles）
- `k`：若分子階數 $\ge$ 分母階數時的直接項（多項式除法的商）

這就是手算部分分式「求 $K_0,K_1,K_2$」步驟的自動化版本。

In [ ]:
pz = zeros(1, n-1);
for i = 1:n-1
    pz(i) = exp(p(i)*T);        % s 域極點 p 對應 z 域極點 exp(p*T)（複數平移概念的體現）
end

[numzz, denomz] = residue(r, pz, kdir);  % 反向呼叫 residue：用「留數+極點」合併回多項式
numz = conv(numzz, [1 0]);               % 乘上一個 z（因為查表結果都帶一個 z，如 z/(z-a)）
Ez_from_s = tf(numz, denomz, T)
% 理論結果：(z^3-1.658z^2+0.6804z)/(z^3-2.489z^2+2.038z-0.5488)

**函數介紹：`residue(r, p, k)`**（反向呼叫）——已知留數與極點，回推合併成一個多項式除以多項式的有理函數，等同於「把部分分式通分」。

**函數介紹：`conv(a, b)`**——計算兩個多項式係數向量的摺積，數學上等同於多項式相乘。這裡用來把多項式乘上 $z$（`[1 0]` 代表多項式 $z$）。

**函數介紹：`tf(num, den, T)`**——多了取樣週期 `T` 參數時，建立的是**離散時間**轉移函數（自變數為 $z$）。

**數學 ↔ MATLAB 對照**：

| 數學 | MATLAB |
|---|---|
| $E(s)$ 部分分式展開，求 $K_0,K_1,K_2$ | `[r,p,k]=residue(num,denom)` |
| $\frac{1}{s+a}\to\frac{z}{z-e^{-aT}}$（逐項查表） | `pz(i)=exp(p(i)*T)` |
| 通分合併成單一有理式 | `residue(r,pz,kdir)` + `conv(numzz,[1 0])` |

In [ ]:
% 方法二：先反拉普拉斯回到時域，代入 t=kT，再取 z 轉換
syms s t
Es_sym = (s^2+4*s+3)/(s^3+6*s^2+8*s);
et = ilaplace(Es_sym);          % 反拉普拉斯轉換：s 域 -> 時域 e(t)
ekT2 = subs(et, t, k*T);        % 代入 t = kT，得到取樣後的數列
Ez_alt = ztrans(ekT2);          % 對數列取 z 轉換
pretty(Ez_alt)

**函數介紹：`ilaplace(expr)`**——對符號表達式做反拉普拉斯轉換，把 $s$ 域函數轉回時域函數 $e(t)$。

**函數介紹：`subs(expr, old, new)`**——把符號表達式中的變數 `old` 代換成 `new`。這裡把時間 `t` 換成 `k*T`，對應「每隔 $T$ 秒取一個樣本」的數學操作。

兩種方法（部分分式查表 vs. 先反拉普拉斯再取樣）殊途同歸，結果一致。

---

## 五、差分方程式的解法 (2.6) —— 例 2.10：逐次代入法

求解 $m(k)=e(k)-e(k-1)-m(k-1),\ k\ge0$，其中 $e(k)$ 在偶數 $k$ 為 1、奇數 $k$ 為 0，$e(-1)=m(-1)=0$。

這是數位電腦求解差分方程式最直接的方式：從 $k=0$ 開始一步步往後代入。

In [ ]:
mkminus1 = 0;   % m(k-1) 的初始值 m(-1)
ekminus1 = 0;   % e(k-1) 的初始值 e(-1)
ek = 1;         % e(0)
for kk = 0:6
    mk = ek - ekminus1 - mkminus1;
    fprintf('k=%d, m(k)=%d\n', kk, mk);
    mkminus1 = mk;       % 這一輪算出的新值，變成下一輪的「舊值」
    ekminus1 = ek;
    ek = 1 - ek;          % e(k) 在 0 與 1 之間交替
end

**函數介紹：`for k = a:b ... end`**——迴圈語法，讓 `k` 依序取 `a, a+1, ..., b`。這裡完美對應差分方程式「逐次代入」的數學過程：每一次迭代就是計算下一個時間點的 $m(k)$。

**數學 ↔ MATLAB 對照**：

- `mkminus1` ↔ $m(k-1)$；`mk` ↔ $m(k)$
- 迴圈本體最後三行賦值 ↔ 把「這一輪算出的新值」變成「下一輪的舊值」，即時間往前推進一步

z 轉換法也能解出相同的結果（見 `chp2.md` 例 2.11），兩種方法互相驗證。

---

## 六、反 z 轉換 (2.7)

四種方法：**冪級數法（長除法）**、**部分分式展開法**、**反演公式法（留數定理）**、**離散摺積法**。這裡示範部分分式法（含重根）與離散摺積法的 MATLAB 實作。

### 部分分式展開法：`iztrans()`

核心技巧是先展開 $\dfrac{E(z)}{z}$（因為查表用的轉換式分子都帶一個 $z$，如 $\dfrac{z}{z-a}\to a^k$），再乘回一個 $z$。

In [ ]:
syms kk_sym
Ez1 = z/((z-1)*(z-2));
iztrans(Ez1, kk_sym)
% 理論結果：e(k) = 2^k - 1

**函數介紹：`iztrans(expr, k)`**——`ztrans()` 的反運算，對符號表達式 `expr`（以 `z` 為自變數）計算反 z 轉換，結果以 `k` 為自變數表示。這一行程式相當於手算部分分式＋查表的整個流程。

In [ ]:
% 例 2.16：重根情形
Ez2 = z/(z-1)^2;
iztrans(Ez2, kk_sym)
% 理論結果：e(k) = k（$z=1$ 處為二重極點，反演公式需要對留數多做一次微分，見 chp2.md）

In [ ]:
% 例 2.14：共軛複數極點 -> residue 找留數與極點，再轉成正弦形式
num14 = [0, 0, -3.894];
den14 = [1, 0, 0.6065];
[r14, p14, k14] = residue(num14, den14);
r14, p14
% 理論結果：p = ±j0.7788（純虛數極點，代表無衰減的振盪）
%           r = ±j2.5001（對應例 2.14 中 k1 = 2.5∠90°）
% 由此可推出 y(k) = -5 * exp(-0.25k) * sin(pi*k/2)（推導見 chp2.md 第六節）

### 離散摺積法：`conv()`

若 $E(z)=E_1(z)E_2(z)$，兩個數列的**離散摺積和** $e(k)=\sum_{n=0}^k e_1(n)e_2(k-n)$ 就是反 z 轉換的結果。`conv()` 這個函數同時扮演「多項式相乘」與「離散摺積」兩個角色——這不是巧合，因為兩者在數學結構上完全相同，這正是 z 轉換把「摺積」變成「乘法」這個核心性質的體現。

In [ ]:
% 例 2.17
e1 = [1 1 1 1 1 1];
e2 = [0 1 2 4 8 16];
e_conv = conv(e1, e2)
% e(3) 應為 7，與例 2.13 用部分分式法得到的 e(k)=-1+2^k 在 k=3 的值 (-1+8=7) 一致

---

## 七、模擬圖與訊號流程圖 (2.8)

離散系統的基本元件是**時間延遲**（延遲一個取樣週期 $T$），其轉移函數為 $z^{-1}$，角色對應類比系統中的**積分器**（轉移函數 $s^{-1}$）。$n$ 階系統若逐項照差分方程式畫模擬圖，需要 $2n$ 個延遲元件（**非最小實現**）；但理論上只需要 $n$ 個延遲元件（狀態）就能完整描述系統（**最小實現**）。這個「如何用最少延遲元件表示系統」的問題，正是引出下一節**狀態變數**方法的動機——本節沒有獨立的程式碼範例，重點在觀念的建立。

---

## 八、狀態變數 (2.9) —— 控制標準型 (CCF)

給定轉移函數：

$$
G(z) = \frac{Y(z)}{U(z)} = \frac{b_{n-1}z^{n-1}+\cdots+b_1z+b_0}{z^n+a_{n-1}z^{n-1}+\cdots+a_1z+a_0}
$$

**控制標準型**（Control Canonical Form）讓我們可以「看一眼轉移函數就直接寫出狀態方程式」：

$$
x(k+1) = \begin{bmatrix}0&1&0&\cdots&0\\0&0&1&\cdots&0\\\vdots&&&\ddots&\vdots\\-a_0&-a_1&-a_2&\cdots&-a_{n-1}\end{bmatrix}x(k) + \begin{bmatrix}0\\0\\\vdots\\1\end{bmatrix}u(k), \qquad y(k) = \begin{bmatrix}b_0&b_1&\cdots&b_{n-1}\end{bmatrix}x(k)
$$

**例 2.19**：$G(z) = \dfrac{z^2+2z+1}{z^3+2z^2+z+0.5}$，即 $b_0=1,b_1=2,b_2=1$，$a_0=0.5,a_1=1,a_2=2$。

In [ ]:
% 例 2.19：控制標準型狀態矩陣
b = [1 2 1];      % b0 b1 b2
a0 = 0.5; a1 = 1; a2 = 2;

A_ccf = [0 1 0; 0 0 1; -a0 -a1 -a2]
B_ccf = [0; 0; 1]
C_ccf = b          % [b0 b1 b2]
D_ccf = 0;

**觀察 $A$ 矩陣結構的規律**：最後一列放 $-a_0,-a_1,\dots,-a_{n-1}$（分母係數取負號），其餘位置是移位矩陣（次對角線全 1）；$C$ 向量直接就是分子係數 $[b_0,b_1,\dots,b_{n-1}]$。這個規律讓我們不需要重新推導，直接「照抄」係數即可建立狀態模型。

---

## 九、相似轉換與對角化 (2.10)

**相似轉換**：引入可逆矩陣 $P$，令 $x(k)=Pw(k)$，可得到另一組等價的狀態模型：

$$
A_w = P^{-1}AP, \qquad B_w = P^{-1}B, \qquad C_w = CP, \qquad D_w = D
$$

**關鍵性質**：相似轉換不改變特徵值、行列式、跡（trace），也不改變轉移函數——因為這些都是系統的「本質特性」，不會因為換了一組內部座標而改變。

### 例 2.23：任選 $P$ 做相似轉換

In [ ]:
A = [0.8 1; 0 0.9];
B = [0; 1];
C = [1 0];

P = [1 -1; 1 1];
Aw = inv(P)*A*P
Bw = inv(P)*B
Cw = C*P

**函數介紹：`inv(M)`**——計算方陣 `M` 的反矩陣 $M^{-1}$，是相似轉換公式 $A_w=P^{-1}AP$ 中 $P^{-1}$ 的數值計算。

**數學 ↔ MATLAB 對照**：$A_w=P^{-1}AP$ ↔ `Aw = inv(P)*A*P`（矩陣乘法用 `*`，順序不可顛倒，因為矩陣乘法沒有交換律）。

In [ ]:
disp('驗證特徵值不變：');
eig(A)
eig(Aw)
% 兩者應完全相同：z1=0.8, z2=0.9

**函數介紹：`eig(A)`**（單一輸出）——只回傳矩陣 `A` 的特徵值（不含特徵向量），對應特徵方程式 $|zI-A|=0$ 的根。

### 例 2.25：用 `[V,D]=eig(A)` 做對角化

若特徵值相異，取相似轉換矩陣 $P=M$（特徵向量組成的模態矩陣），可以得到對角化的狀態模型 $\Lambda=M^{-1}AM$。

In [ ]:
[M, LAMBDA] = eig(A)

**函數介紹：`[V, D] = eig(A)`**（雙輸出）——計算方陣 `A` 的特徵向量與特徵值。`V` 的每一行是一個特徵向量，`D` 是以特徵值為對角元素的對角矩陣，滿足 $A\cdot V = V\cdot D$。

In [ ]:
disp('驗證 A*M - M*LAMBDA 應接近零矩陣：');
A*M - M*LAMBDA

**驗證方式**：`A*V - V*D` 若（數值上）接近零矩陣，就代表 `eig()` 的結果滿足特徵值方程式 $AM=M\Lambda$。

> **注意**：MATLAB/Octave 的 `eig()` 回傳的特徵向量通常會做單位化（長度歸一），數值上可能和手算「任取比例常數=1」的結果差一個縮放係數，但代表的是同一個特徵方向，不影響對角化後的特徵值 $\Lambda$ 本身。

---

## 十、由狀態方程式求轉移函數 (2.11)

公式：

$$
G(z) = C[zI-A]^{-1}B+D
$$

推導自對狀態方程式取 z 轉換、解出 $X(z)$、代入輸出方程式（詳見 `chp2.md` 第十節）。MATLAB 中最直接的做法是用 `ss2tf()`。

### 例 2.28

In [ ]:
A28 = [1.35 0.55; -0.45 0.35];
B28 = [0.5; 0.5];
C28 = [1 -1];
D28 = 0;
T28 = 1;

[num28, den28] = ss2tf(A28, B28, C28, D28);
Gz28 = tf(num28, den28, T28)
% 理論結果：1/(z^2-1.7z+0.72)

**函數介紹：`ss2tf(A,B,C,D)`**——直接把狀態空間矩陣 $(A,B,C,D)$ 轉換成轉移函數的分子、分母係數向量，內部原理正是 $G(z)=C[zI-A]^{-1}B+D$ 的數值化實作。

**數學 ↔ MATLAB 對照**：

| 數學 | MATLAB |
|---|---|
| $zI-A$ | 內部隱含於 `ss2tf` |
| $[zI-A]^{-1}$ | 內部隱含於 `ss2tf` |
| $C[zI-A]^{-1}B+D$ | `[num,den] = ss2tf(A,B,C,D)` |

也可以用符號運算 `syms z; Gz = C*inv(z*eye(2)-A)*B+D; simplify(Gz)` 手動走一遍公式（見 `chp2.md`），兩者結果一致，但 `ss2tf` 更快速、適合放進程式流程中。

---

## 十一、狀態方程式的解 (2.12)

一般解：

$$
x(k) = \Phi(k)x(0) + \sum_{j=0}^{k-1}\Phi(k-1-j)Bu(j), \qquad \Phi(k) \triangleq A^k
$$

$\Phi(k)$ 稱為**狀態轉移矩陣**（state transition matrix）。可以用**遞迴法**（電腦逐步代入，例 2.29）求數值解，或用 **z 轉換法**（$\Phi(k)=\mathcal{Z}^{-1}[z[zI-A]^{-1}]$）求 $\Phi(k)$ 的封閉公式（例 2.30，見 `chp2.md`）。

### 例 2.29：遞迴解

In [ ]:
A29 = [0 1; -2 -3];
B29 = [0; 1];
C29 = [3 1];
x = [0; 0];
u = 1;
for kk = 0:5
    x1 = A29*x + B29*u;
    y = C29*x;
    fprintf('k=%d, y(k)=%g\n', kk, y);
    x = x1;   % 這次算出的新狀態，變成下一輪迭代的「目前狀態」
end
% 理論結果：y = 0 1 1 -1 5 -9

**數學 ↔ MATLAB 對照**：迴圈內 `x1 = A29*x + B29*u` 正是遞迴公式 $x(k+1)=Ax(k)+Bu(k)$ 的直接數值實作，完全不需要事先求出 $\Phi(k)$ 的封閉公式。

### 例 2.31：狀態轉移矩陣的性質

$$
\Phi(0)=I, \qquad \Phi(k_1+k_2)=\Phi(k_1)\Phi(k_2), \qquad \Phi(-k)=\Phi^{-1}(k)
$$

In [ ]:
A31 = [1 0; 0 0.5];
Phi = @(kk) A31^kk;    % 注意：^ 是矩陣次方，不是逐元素次方 .^

disp('Phi(0) 應為單位矩陣：');
Phi(0)

**⚠️ 特別注意**：`A^k` 代表矩陣 $A$ 自乘 $k$ 次（矩陣次方）；若誤寫成 `A.^k`（逐元素次方），MATLAB/Octave 會把矩陣中每個元素各自獨立做 $k$ 次方，結果完全不同，也不再具有 $\Phi(k)=A^k$ 的物理意義。

In [ ]:
k1 = 2; k2 = 3;
disp('Phi(k1+k2) - Phi(k1)*Phi(k2) 應接近零矩陣：');
Phi(k1+k2) - Phi(k1)*Phi(k2)

驗證通過（結果為零矩陣），代表匿名函數 `Phi` 確實滿足狀態轉移矩陣的指數律性質 $\Phi(k_1+k_2)=\Phi(k_1)\Phi(k_2)$。

**函數介紹：`@(kk) A31^kk`**——MATLAB/Octave 的**匿名函數**（anonymous function）語法，`@(參數) 表達式` 定義一個以 `kk` 為輸入、回傳 `A31^kk` 的函數，可以像一般函數一樣呼叫（如 `Phi(0)`、`Phi(5)`），方便重複計算不同 $k$ 值的 $\Phi(k)=A^k$。

---

## 十二、線性時變系統 (2.13)

若系統矩陣 $A(k),B(k),C(k),D(k)$ 隨時間變化，狀態轉移矩陣需要改為：

$$
\Phi(k,k_0) = A(k-1)A(k-2)\cdots A(k_0) = \prod_{j=k_0}^{k-1}A(j)
$$

必須在每個時間點重新計算，不能像非時變系統一樣直接套用 $A^k$。當 $A$ 不再是 $k$ 的函數時，$\Phi(k,k_0)=A^{k-k_0}$，恰好化簡回非時變系統的公式——**非時變系統只是時變系統的特例**。本節純屬觀念延伸，教材未提供獨立數值範例。

---

## 十三、本章總結

本 Notebook 完整走過了：z 轉換的定義與性質、由 s 域函數求 z 轉換、差分方程式的三種解法、反 z 轉換的四種方法、模擬圖與訊號流程圖、狀態變數模型的建立（控制標準型）、相似轉換與對角化、由狀態方程式反推轉移函數、以及狀態方程式的解（狀態轉移矩陣）。這些工具是後續章節（取樣重建、開迴路/閉迴路分析、穩定性、數位控制器設計）的數學基礎。

完整理論推導與所有訂正說明請見 `chp2.md`；純程式碼精簡版請見 `chp2.m`。